# Finance MLOps: Exploratory Data Analysis & Feature Engineering

Notebook ini melakukan proses muat data dari MongoDB, analisis tren harga, dan *Feature Engineering* (membuat indikator teknikal).

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ta

sys.path.append(os.path.join(os.getcwd(), '..', 'data_pipeline', 'src'))
from db_helper import MongoDBHelper

## 1. Load Data dari MongoDB

In [ ]:
db = MongoDBHelper()
db.connect()
data = db.get_all_data('stock_data', 'BBCA.JK')
df = pd.DataFrame(data)
db.close()

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
if '_id' in df.columns:
    df.drop('_id', axis=1, inplace=True)

df.tail()

## 2. Feature Engineering (Technical Indicators)

In [ ]:
df['SMA_14'] = ta.trend.sma_indicator(df['Close'], window=14)
df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)
df['RSI_14'] = ta.momentum.rsi(df['Close'], window=14)
macd = ta.trend.MACD(df['Close'])
df['MACD'] = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
bb = ta.volatility.BollingerBands(df['Close'])
df['BB_High'] = bb.bollinger_hband()
df['BB_Low'] = bb.bollinger_lband()
df['Daily_Return'] = df['Close'].pct_change()
df.dropna(inplace=True)

df.tail()

## 3. Visualization

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close Price', color='blue', alpha=0.6)
plt.plot(df.index, df['SMA_14'], label='SMA 14', color='red', alpha=0.8)
plt.plot(df.index, df['SMA_50'], label='SMA 50', color='green', alpha=0.8)
plt.title('BBCA.JK Closing Price & Moving Averages')
plt.legend()
plt.grid(True)
plt.show()